# Delta Table - All Data Types with Random Timezones
Creates a Delta table demonstrating all major Spark data types. The **TIMESTAMP** column includes random timezone offsets to show how Spark normalizes them to UTC on storage, while **TIMESTAMP_NTZ** remains unaffected by timezone.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    ShortType, ByteType, FloatType, DoubleType, DecimalType,
    BooleanType, DateType, TimestampType, TimestampNTZType,
    BinaryType, ArrayType, MapType
)
from decimal import Decimal
import random
import string
import datetime

In [ ]:
# Define schema with a wide range of data types

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("byte_col", ByteType(), True),
    StructField("short_col", ShortType(), True),
    StructField("int_col", IntegerType(), True),
    StructField("long_col", LongType(), True),
    StructField("float_col", FloatType(), True),
    StructField("double_col", DoubleType(), True),
    StructField("decimal_col", DecimalType(38, 18), True),
    StructField("string_col", StringType(), True),
    StructField("boolean_col", BooleanType(), True),
    StructField("date_col", DateType(), True),
    StructField("timestamp_col", TimestampType(), True),
    StructField("timestamp_ntz_col", TimestampNTZType(), True),
    StructField("timezone_used", StringType(), True),
    StructField("binary_col", BinaryType(), True),
    StructField("array_col", ArrayType(IntegerType()), True),
    StructField("map_col", MapType(StringType(), IntegerType()), True),
])

In [ ]:
# Generate 100 rows with random timezones applied to timestamps

random.seed(42)

# Pool of timezone offsets (hours from UTC) representing real-world zones
TIMEZONE_OFFSETS = [
    (-12, "UTC-12:00 (Baker Island)"),
    (-10, "UTC-10:00 (Hawaii)"),
    (-8, "UTC-08:00 (US Pacific)"),
    (-7, "UTC-07:00 (US Mountain)"),
    (-6, "UTC-06:00 (US Central)"),
    (-5, "UTC-05:00 (US Eastern)"),
    (-4, "UTC-04:00 (Atlantic)"),
    (-3, "UTC-03:00 (Brazil)"),
    (0, "UTC+00:00 (London/UTC)"),
    (1, "UTC+01:00 (Paris/Berlin)"),
    (2, "UTC+02:00 (Cairo/Johannesburg)"),
    (3, "UTC+03:00 (Moscow)"),
    (4, "UTC+04:00 (Dubai)"),
    (5, "UTC+05:00 (Karachi)"),
    (5.5, "UTC+05:30 (India)"),
    (8, "UTC+08:00 (Singapore/Beijing)"),
    (9, "UTC+09:00 (Tokyo)"),
    (9.5, "UTC+09:30 (Adelaide)"),
    (10, "UTC+10:00 (Sydney)"),
    (12, "UTC+12:00 (Auckland)"),
    (13, "UTC+13:00 (Samoa)"),
]

def random_string(length=10):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

def random_timestamp_with_tz():
    """Generate a random timezone-aware timestamp with full microsecond precision.
    Returns (datetime_utc, datetime_naive_local, timezone_label).
    
    Spark TIMESTAMP stores as UTC - the timezone offset shifts the value on storage.
    """
    year = random.randint(2020, 2026)
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    hour = random.randint(0, 23)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    microsecond = random.randint(0, 999999)
    
    # Pick a random timezone
    offset_hours, tz_label = random.choice(TIMEZONE_OFFSETS)
    
    # Create the local datetime with timezone info
    offset = datetime.timezone(datetime.timedelta(hours=offset_hours))
    local_dt = datetime.datetime(year, month, day, hour, minute, second, microsecond, tzinfo=offset)
    
    # Convert to UTC (this is what Spark TIMESTAMP stores)
    utc_dt = local_dt.astimezone(datetime.timezone.utc).replace(tzinfo=None)
    
    # The naive local time (for TIMESTAMP_NTZ - no conversion)
    naive_local = datetime.datetime(year, month, day, hour, minute, second, microsecond)
    
    return utc_dt, naive_local, tz_label

def random_binary():
    return bytes(random.getrandbits(8) for _ in range(random.randint(4, 16)))

rows = []
for i in range(1, 101):
    ts_utc, ts_naive, tz_label = random_timestamp_with_tz()
    
    row = (
        i,                                                          # id
        random.randint(-128, 127),                                  # byte_col
        random.randint(-32768, 32767),                              # short_col
        random.randint(-2147483648, 2147483647),                    # int_col
        random.randint(-9223372036854775808, 9223372036854775807),  # long_col
        random.uniform(-1e6, 1e6),                                 # float_col
        random.uniform(-1e15, 1e15),                                # double_col
        Decimal(format(random.uniform(-1e10, 1e10), '.18f')),       # decimal_col
        random_string(random.randint(5, 50)),                       # string_col
        random.choice([True, False]),                               # boolean_col
        datetime.date(
            random.randint(2000, 2026),
            random.randint(1, 12),
            random.randint(1, 28)
        ),                                                          # date_col
        ts_utc,                                                     # timestamp_col (UTC-normalized)
        ts_naive,                                                   # timestamp_ntz_col (local time, no tz)
        tz_label,                                                   # timezone_used (for reference)
        random_binary(),                                            # binary_col
        [random.randint(1, 1000) for _ in range(random.randint(1, 5))],  # array_col
        {random_string(3): random.randint(1, 100) for _ in range(random.randint(1, 3))},  # map_col
    )
    rows.append(row)

print(f"Generated {len(rows)} rows")
print(f"\nSample row 1:")
print(f"  Timezone used:      {rows[0][13]}")
print(f"  timestamp_col (UTC): {rows[0][11]} (microseconds: {rows[0][11].microsecond})")
print(f"  timestamp_ntz_col:   {rows[0][12]} (microseconds: {rows[0][12].microsecond})")
print(f"\nSample row 2:")
print(f"  Timezone used:      {rows[1][13]}")
print(f"  timestamp_col (UTC): {rows[1][11]} (microseconds: {rows[1][11].microsecond})")
print(f"  timestamp_ntz_col:   {rows[1][12]} (microseconds: {rows[1][12].microsecond})")

In [ ]:
# Create DataFrame and inspect

df = spark.createDataFrame(rows, schema=schema)

print("=== Schema ===")
df.printSchema()

print("\n=== Sample: Timestamp vs Timestamp_NTZ with timezone context ===")
print("Notice how the SAME local time produces DIFFERENT UTC values based on timezone,")
print("while TIMESTAMP_NTZ always stores the local time as-is.\n")
df.select(
    "id", "timezone_used", "timestamp_col", "timestamp_ntz_col"
).show(15, truncate=False)

In [ ]:
# Write to Delta table

TABLE_NAME = "all_datatypes_demo_with_tz"

df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Delta table '{TABLE_NAME}' created successfully with {df.count()} rows.")

In [ ]:
# Verify precision and timezone effects

print("=== Reading back from Delta table ===\n")
df_read = spark.read.table(TABLE_NAME)
df_read.printSchema()

print("\n=== Verifying microsecond precision with timezone context ===")
print("TIMESTAMP: value was converted from local time to UTC using the random timezone.")
print("TIMESTAMP_NTZ: value is the original local time, timezone has no effect.\n")

spark.sql(f"""
    SELECT 
        id,
        timezone_used,
        date_format(timestamp_col, 'yyyy-MM-dd HH:mm:ss.SSSSSS') as timestamp_utc,
        date_format(timestamp_ntz_col, 'yyyy-MM-dd HH:mm:ss.SSSSSS') as timestamp_ntz_local
    FROM {TABLE_NAME}
    ORDER BY id
    LIMIT 20
""").show(truncate=False)